In [1]:
import matplotlib.pyplot as plt
import pickle

from pmbrl.model2 import Model
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
nome_do_arquivo = 'triplet.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    model = exp['model']

del exp
del arquivo

In [3]:
# data.evaluate_model(model, path='../testing_data(TEST).csv')
data.evaluate_model(model, path='../testing_data.csv')
results = data.get_evaluation_metrics()
results.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,rse_s1,rse_s2,rse_s3,rse_r,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized,rse,rse_normalized
0,0,0,"(0.5280280669895349, 0.1691488904272192)","(-0.018, -0.046, 0.013, -0.019)",0,1.0,"(-0.019, -0.285, 0.013, 1.258)",0.0,1.0,"(-0.025, -0.525, 0.038, 2.534)",...,0.014,0.022,0.168,0.0,0.550158,0.529499,0.553957,0.449046,0.208,0.520665
1,1,0,"(0.5280280669895349, 0.1691488904272192)","(-0.019, -0.285, 0.013, 1.258)",0,1.0,"(-0.025, -0.525, 0.038, 2.534)",1.0,1.0,"(-0.035, -0.288, 0.088, 1.326)",...,0.050,0.022,0.121,0.0,0.565998,0.538348,0.553957,0.445024,0.212,0.525832
2,2,0,"(0.5280280669895349, 0.1691488904272192)","(-0.025, -0.525, 0.038, 2.534)",1,1.0,"(-0.035, -0.288, 0.088, 1.326)",0.0,1.0,"(-0.041, -0.532, 0.115, 2.694)",...,0.000,0.020,0.194,0.0,0.552270,0.526057,0.549161,0.451271,0.220,0.519690
3,3,0,"(0.5280280669895349, 0.1691488904272192)","(-0.035, -0.288, 0.088, 1.326)",0,1.0,"(-0.041, -0.532, 0.115, 2.694)",1.0,1.0,"(-0.052, -0.3, 0.169, 1.601)",...,0.030,0.007,0.045,0.0,0.561774,0.533432,0.517986,0.438521,0.097,0.512928
4,4,0,"(0.5280280669895349, 0.1691488904272192)","(-0.041, -0.532, 0.115, 2.694)",1,1.0,"(-0.052, -0.3, 0.169, 1.601)",0.0,1.0,"(-0.058, -0.546, 0.201, 3.045)",...,0.003,0.017,0.055,0.0,0.549102,0.526794,0.541966,0.439377,0.078,0.514310


In [4]:
expansions = {
    'estimated_p': ['estimated_p0', 'estimated_p1'],
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    's__': ['s__0', 's__1', 's__2', 's__3'],
}


# df = data.evaluation_data[data.evaluation_data['episode'] == 98].copy().reset_index(drop=True)
df = data.evaluation_data.copy()
df = get_data_expanded(df, expansions)
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s2,s3,s_0,s_1,s_2,s_3,s__0,s__1,s__2,s__3
0,0,0,"(0.5280280669895349, 0.1691488904272192)","(-0.018, -0.046, 0.013, -0.019)",0,1.0,"(-0.019, -0.285, 0.013, 1.258)",0.0,1.0,"(-0.025, -0.525, 0.038, 2.534)",...,0.013,-0.019,-0.019,-0.285,0.013,1.258,-0.025,-0.525,0.038,2.534
1,1,0,"(0.5280280669895349, 0.1691488904272192)","(-0.019, -0.285, 0.013, 1.258)",0,1.0,"(-0.025, -0.525, 0.038, 2.534)",1.0,1.0,"(-0.035, -0.288, 0.088, 1.326)",...,0.013,1.258,-0.025,-0.525,0.038,2.534,-0.035,-0.288,0.088,1.326
2,2,0,"(0.5280280669895349, 0.1691488904272192)","(-0.025, -0.525, 0.038, 2.534)",1,1.0,"(-0.035, -0.288, 0.088, 1.326)",0.0,1.0,"(-0.041, -0.532, 0.115, 2.694)",...,0.038,2.534,-0.035,-0.288,0.088,1.326,-0.041,-0.532,0.115,2.694
3,3,0,"(0.5280280669895349, 0.1691488904272192)","(-0.035, -0.288, 0.088, 1.326)",0,1.0,"(-0.041, -0.532, 0.115, 2.694)",1.0,1.0,"(-0.052, -0.3, 0.169, 1.601)",...,0.088,1.326,-0.041,-0.532,0.115,2.694,-0.052,-0.300,0.169,1.601
4,4,0,"(0.5280280669895349, 0.1691488904272192)","(-0.041, -0.532, 0.115, 2.694)",1,1.0,"(-0.052, -0.3, 0.169, 1.601)",0.0,1.0,"(-0.058, -0.546, 0.201, 3.045)",...,0.115,2.694,-0.052,-0.300,0.169,1.601,-0.058,-0.546,0.201,3.045


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

def find_params(m, inputs, target, initial_params, num_epochs=5000, learning_rate=.01):
    input_values = torch.tensor(inputs).reshape(1, len(inputs))
    param = torch.tensor(initial_params, requires_grad=True)
    target_value = torch.tensor(target).reshape(1, len(target))

    optimizer = optim.Adam([param], lr=learning_rate)
    criterion = nn.MSELoss(reduction='none')

    history = []

    for p in m.parameters():
        p.requires_grad = False

    for epoch in range(num_epochs):
        # Forward pass
        state_inputs = torch.concat([input_values, param.reshape(1, len(initial_params))], dim=1)
        output = m(state_inputs.float())  # Add batch dimension

        # Calculate the loss
        open_loss = criterion(output.float(), target_value.float())
        loss = torch.sqrt(open_loss.sum(axis=1).mean())
        

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append((param.tolist(), output.tolist()[0], loss.item()))

        if (epoch + 1) % 100 == 0:
            # print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {param.item():.4f}, Output: {output.item():.4f}')
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')
        return history


In [6]:
def optimize(row):
    m = model.transition_estimator.state_layer
    inputs = [row.s0, row.s1, row.s2, row.s3, row.a]
    targets = [row.s_0, row.s_1, row.s_2, row.s_3]
    init_param = [row.estimated_p0, row.estimated_p1]

    hist = find_params(m, inputs, targets, init_param)
    return hist[-1]


In [7]:
df[['new_estimated_p', 'new_estimated_s', 'param_rse']] = df.apply(lambda row: optimize(row), axis=1, result_type='expand')

In [8]:
df[['p', 'estimated_p', 'new_estimated_p']].head()

,p,estimated_p,new_estimated_p
0,"(0.5280280669895349, 0.1691488904272192)","(0.982, -3.454)","[0.972000002861023, -3.444000005722046]"
1,"(0.5280280669895349, 0.1691488904272192)","(0.773, -3.32)","[0.7630000114440918, -3.309999942779541]"
2,"(0.5280280669895349, 0.1691488904272192)","(0.731, -4.067)","[0.7409999966621399, -4.077000141143799]"
3,"(0.5280280669895349, 0.1691488904272192)","(0.766, -3.666)","[0.7559999823570251, -3.6559998989105225]"
4,"(0.5280280669895349, 0.1691488904272192)","(0.461, -3.871)","[0.47099998593330383, -3.88100004196167]"


In [9]:
expansions = {
    'new_estimated_p': ['new_estimated_p0', 'new_estimated_p1'],
}

final_df = data.evaluation_data.copy()
final_df = get_data_expanded(df, expansions)
final_df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s_3,s__0,s__1,s__2,s__3,new_estimated_p,new_estimated_s,param_rse,new_estimated_p0,new_estimated_p1
0,0,0,"(0.5280280669895349, 0.1691488904272192)","(-0.018, -0.046, 0.013, -0.019)",0,1.0,"(-0.019, -0.285, 0.013, 1.258)",0.0,1.0,"(-0.025, -0.525, 0.038, 2.534)",...,1.258,-0.025,-0.525,0.038,2.534,"[0.972000002861023, -3.444000005722046]","[-0.02947138249874115, -0.3243144750595093, 0....",0.043276,0.972,-3.444
1,1,0,"(0.5280280669895349, 0.1691488904272192)","(-0.019, -0.285, 0.013, 1.258)",0,1.0,"(-0.025, -0.525, 0.038, 2.534)",1.0,1.0,"(-0.035, -0.288, 0.088, 1.326)",...,2.534,-0.035,-0.288,0.088,1.326,"[0.7630000114440918, -3.309999942779541]","[-0.023772403597831726, -0.509128212928772, 0....",0.067673,0.763,-3.310
2,2,0,"(0.5280280669895349, 0.1691488904272192)","(-0.025, -0.525, 0.038, 2.534)",1,1.0,"(-0.035, -0.288, 0.088, 1.326)",0.0,1.0,"(-0.041, -0.532, 0.115, 2.694)",...,1.326,-0.041,-0.532,0.115,2.694,"[0.7409999966621399, -4.077000141143799]","[-0.015354827046394348, -0.32026541233062744, ...",0.072719,0.741,-4.077
3,3,0,"(0.5280280669895349, 0.1691488904272192)","(-0.035, -0.288, 0.088, 1.326)",0,1.0,"(-0.041, -0.532, 0.115, 2.694)",1.0,1.0,"(-0.052, -0.3, 0.169, 1.601)",...,2.694,-0.052,-0.300,0.169,1.601,"[0.7559999823570251, -3.6559998989105225]","[-0.0353735089302063, -0.5212457776069641, 0.0...",0.111432,0.756,-3.656
4,4,0,"(0.5280280669895349, 0.1691488904272192)","(-0.041, -0.532, 0.115, 2.694)",1,1.0,"(-0.052, -0.3, 0.169, 1.601)",0.0,1.0,"(-0.058, -0.546, 0.201, 3.045)",...,1.601,-0.058,-0.546,0.201,3.045,"[0.47099998593330383, -3.88100004196167]","[-0.040719859302043915, -0.334219753742218, 0....",0.140765,0.471,-3.881


In [10]:
def predict(row):
    with torch.no_grad():
        m = model.transition_estimator.state_layer
        inputs = [row.s_0, row.s_1, row.s_2, row.s_3, row.a_]
        params = [row.new_estimated_p0, row.new_estimated_p1]
        # params = [row.estimated_p0, row.estimated_p1]

        input_values = torch.tensor(inputs).reshape(1, len(inputs))
        param = torch.tensor(params)

        for p in m.parameters():
            p.requires_grad = False

        state_inputs = torch.concat([input_values, param.reshape(1, len(params))], dim=1)
        output = m(state_inputs.float())  # Add batch dimension

    return [round(v, 3) for v in output.tolist()[0]]


In [11]:
final_df['new_estimated_s'] = final_df.apply(lambda row: predict(row), axis=1)

In [12]:
df_metrics = final_df.copy()
df_metrics['estimated_s'] = df_metrics['new_estimated_s']
results = data.get_evaluation_metrics(df_metrics)

In [13]:
final_df[['p', 'estimated_p', 'new_estimated_p']]

,p,estimated_p,new_estimated_p
0,"(0.5280280669895349, 0.1691488904272192)","(0.982, -3.454)","[0.972000002861023, -3.444000005722046]"
1,"(0.5280280669895349, 0.1691488904272192)","(0.773, -3.32)","[0.7630000114440918, -3.309999942779541]"
2,"(0.5280280669895349, 0.1691488904272192)","(0.731, -4.067)","[0.7409999966621399, -4.077000141143799]"
3,"(0.5280280669895349, 0.1691488904272192)","(0.766, -3.666)","[0.7559999823570251, -3.6559998989105225]"
4,"(0.5280280669895349, 0.1691488904272192)","(0.461, -3.871)","[0.47099998593330383, -3.88100004196167]"
...,...,...,...
2122,"(0.0368536405463078, 0.7035040297764357)","(0.178, 0.856)","[0.1679999977350235, 0.8659999966621399]"
2123,"(0.0368536405463078, 0.7035040297764357)","(0.195, 0.918)","[0.20499998331069946, 0.9279966354370117]"
2124,"(0.0368536405463078, 0.7035040297764357)","(0.163, 0.856)","[0.15299999713897705, 0.8659999966621399]"
2125,"(0.0368536405463078, 0.7035040297764357)","(0.16, 0.875)","[0.15000000596046448, 0.8849999904632568]"


In [14]:
final_df[['s__', 'estimated_s', 'new_estimated_s']]

,s__,estimated_s,new_estimated_s
0,"(-0.025, -0.525, 0.038, 2.534)","(-0.021, -0.511, 0.016, 2.702)","[-0.021, -0.511, 0.016, 2.696]"
1,"(-0.035, -0.288, 0.088, 1.326)","(-0.016, -0.338, 0.11, 1.447)","[-0.017, -0.338, 0.11, 1.452]"
2,"(-0.041, -0.532, 0.115, 2.694)","(-0.035, -0.532, 0.095, 2.888)","[-0.035, -0.532, 0.095, 2.893]"
3,"(-0.052, -0.3, 0.169, 1.601)","(-0.037, -0.33, 0.176, 1.646)","[-0.037, -0.33, 0.176, 1.651]"
4,"(-0.058, -0.546, 0.201, 3.045)","(-0.055, -0.543, 0.184, 3.1)","[-0.055, -0.543, 0.184, 3.106]"
...,...,...,...
2122,"(-0.105, -0.761, 0.128, 0.931)","(-0.117, -0.783, 0.123, 0.925)","[-0.117, -0.784, 0.124, 0.928]"
2123,"(-0.12, -0.952, 0.146, 1.156)","(-0.103, -0.953, 0.15, 1.172)","[-0.103, -0.952, 0.15, 1.172]"
2124,"(-0.139, -1.144, 0.169, 1.383)","(-0.121, -1.149, 0.176, 1.412)","[-0.121, -1.15, 0.176, 1.409]"
2125,"(-0.162, -1.336, 0.197, 1.615)","(-0.143, -1.343, 0.206, 1.651)","[-0.143, -1.344, 0.207, 1.647]"


In [15]:
results[['rse', 'rse_normalized', 
       'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 'rse_r', 'rse_s0_normalized',
       'rse_s1_normalized', 'rse_s2_normalized', 'rse_s3_normalized']].describe()

,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_r,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized
count,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.0,2127.000000,2127.000000,2127.000000,2127.000000
mean,0.045927,0.508172,0.007174,0.010742,0.005281,0.022730,0.0,0.553510,0.528698,0.513863,0.436616
std,0.050897,0.005129,0.008066,0.014733,0.004845,0.035952,0.0,0.008518,0.003622,0.011618,0.003076
min,0.003000,0.502527,0.000000,0.000000,0.000000,0.000000,0.0,0.545935,0.526057,0.501199,0.434671
25%,0.020000,0.505197,0.002000,0.003000,0.002000,0.006000,0.0,0.548046,0.526794,0.505995,0.435184
50%,0.031000,0.506993,0.005000,0.006000,0.004000,0.013000,0.0,0.551214,0.527532,0.510791,0.435783
75%,0.053000,0.509345,0.010000,0.012000,0.007000,0.027000,0.0,0.556494,0.529007,0.517986,0.436981
max,0.545000,0.566226,0.120000,0.189000,0.058000,0.388000,0.0,0.672650,0.572517,0.640288,0.467870


In [16]:
del model
del data